# Assignment 1 - Matrix multiplication in Numba


We consider the problem of evaluating the matrix multiplication $C = A\times B$ for matrices $A, B\in\mathbb{R}^{n\times n}$.
A simple Python implementation of the matrix-matrix product is given below through the function `matrix_product`. At the end this
function is checked against the Numpy implementation of the matrix-matrix product.

In [1]:
import numpy as np
import random

def matrix_product(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""
    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in range(m):
        for col_ind in range(n):
            for k in range(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c




In [2]:
%%timeit
a = np.random.randn(10, 10)
b = np.random.randn(10, 10)

c_actual = matrix_product(a, b)
c_expected = a @ b

error = np.linalg.norm(c_actual - c_expected) / np.linalg.norm(c_expected)
print(f"The error is {error}.")

Streaming output truncated to the last 5000 lines.
The error is 1.0986551001325492e-16.
The error is 1.0106317927286941e-16.
The error is 1.4005794842849318e-16.
The error is 1.3973455597789557e-16.
The error is 1.0321195570948424e-16.
The error is 1.0239904258039443e-16.
The error is 1.331439521300072e-16.
The error is 1.2945755260926252e-16.
The error is 1.056149250459942e-16.
The error is 1.0283229645155293e-16.
The error is 7.691226430663381e-17.
The error is 1.2344173600395433e-16.
The error is 1.0789249668632641e-16.
The error is 1.4821504895182114e-16.
The error is 1.0559102468185673e-16.
The error is 1.4187555841354046e-16.
The error is 1.2129117769479422e-16.
The error is 1.2299418937209096e-16.
The error is 1.5077396626182683e-16.
The error is 1.2073949604447019e-16.
The error is 1.0073083012701272e-16.
The error is 1.2178406155266412e-16.
The error is 1.332666651024798e-16.
The error is 1.0527956410498341e-16.
The error is 1.0816056960215129e-16.
The error is 1.3462728919614

The matrix product is one of the most fundamental operations on modern computers. Most algorithms eventually make use of this operation. A lot of effort is therefore spent on optimising the matrix product. Vendors provide hardware optimised BLAS (Basis Linear Algebra Subroutines) that provide highly efficient versions of the matrix product. Alternatively, open-source libraries sucha as Openblas provide widely used generic open-source implementations of this operation.

In this assignment we want to learn at the example of matrix-matrix products about the possible speedups offered by Numba, and the effects of cache-efficient programming.

## 1.1 Benchmark
Benchmark the above function against the Numpy dot product for matrix sizes up to 1000. Plot the timing results of the above function against the timing results for the Numpy dot product. You need not benchmark every dimension up to 1000. Figure out what dimensions to use so that you can represent the result without spending too much time waiting for the code to finish. To perform benchmarks you can use the `%timeit` magic command. An example is

    ```
    timeit_result = %timeit -o matrix_product(a, b)
    print(timeit_result.best)
    ```

## 1.2 Optimize
Now optimise the code by using Numba to JIT-compile it. Also, there is lots of scope for parallelisation in the code. You can for example parallelize the outer-most for-loop. Benchmark the JIT-compiled serial code against the JIT-compiled parallel code. Comment on the expected performance on your system against the observed performance.

## 1.3 (Optional) Cache Optimization
Now let us improve Cache efficiency. Notice that in the matrix $B$ we traverse by columns. However, the default storage ordering in Numpy is row-based. Hence, the expression `mat_b[k, col_ind]` jumps in memory by `n` units if we move from $k$ to $k+1$. Run your parallelized JIT-compiled Numba code again. But this time choose a matrix $B$ that is stored in column-major order. To change an array to column major order you can use the command `np.asfortranarray`.






In [3]:
# 1.2
import numba as nb
from numba import jit
import numpy as np


@jit(nopython=True, parallel=True)
def matrix_product_2(mat_a, mat_b):
    """Returns the product of the matrices mat_a and mat_b."""
    m = mat_a.shape[0]
    n = mat_b.shape[1]

    assert(mat_a.shape[1] == mat_b.shape[0])

    ncol = mat_a.shape[1]

    mat_c = np.zeros((m, n), dtype=np.float64)

    for row_ind in nb.prange(m):
        for col_ind in range(n):
            for k in range(ncol):
                mat_c[row_ind, col_ind] += mat_a[row_ind, k] * mat_b[k, col_ind]

    return mat_c


In [4]:
%%timeit

a = np.random.randn(10, 10)
b = np.random.randn(10, 10)

c_actual = matrix_product_2(a, b)
c_expected = a @ b

error = np.linalg.norm(c_actual - c_expected) / np.linalg.norm(c_expected)
print(f"The error is {error}.")


The error is 1.1498269760450492e-16.
The error is 1.3021565517725418e-16.
The error is 1.752421395752432e-16.
The error is 1.157774095856596e-16.
The error is 1.4931269693022355e-16.
The error is 1.2269157768225642e-16.
The error is 1.3924431890211672e-16.
The error is 1.162834902920001e-16.
124 µs ± 76.9 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)
